# 03 — Handling Class Imbalance with SMOTE

The `at_risk` outcome is imbalanced (more not-at-risk than at-risk trainees). This notebook applies SMOTE (Synthetic Minority Oversampling Technique) to the **training set only**, retrains the XGBoost model on the balanced data, and compares performance against the imbalanced baseline from notebook 02.

**Input:** `data/train_processed.csv`, `data/test_processed.csv`, `models/best_params.json`

**Output:** `models/xgb_smote.json`, `data/train_smote.csv`

**Package versions assumed:** imbalanced-learn==0.11.0, xgboost==1.7.6

In [1]:
import json
import pandas as pd
import numpy as np
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

RANDOM_SEED = 42
TARGET_COL = "at_risk"

## 1. Load processed data and check class balance

In [2]:
train_df = pd.read_csv("data/train_processed.csv")
test_df = pd.read_csv("data/test_processed.csv")

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]
X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("Before SMOTE - class distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

Before SMOTE - class distribution:
at_risk
0    185
1     95
Name: count, dtype: int64
at_risk
0    0.660714
1    0.339286
Name: proportion, dtype: float64


## 2. Apply SMOTE

SMOTE is applied **only** to the training data. The test set remains untouched and reflects the real-world class distribution, which is essential for an unbiased evaluation in notebook 05.

In [3]:
smote = SMOTE(random_state=RANDOM_SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE - class distribution:")
print(pd.Series(y_train_smote).value_counts())
print(pd.Series(y_train_smote).value_counts(normalize=True))
print("\nShape before:", X_train.shape, "-> after:", X_train_smote.shape)

c:\Users\Nahnah\OneDrive\Documents\Project_Work\dummy_smoke_test_package\pipeline_test\venv\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Nahnah\OneDrive\Documents\Project_Work\dummy_smoke_test_package\pipeline_test\venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\Nahnah\AppData\Local\Programs\Python\Python38\lib\subprocess.py", line 493, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\Nahnah\AppData\Local\Programs\Python\Python38\lib\subprocess.py", line 858, in __init__
    self._execute_child(args, executable, preexec_fn, close_fd

After SMOTE - class distribution:
at_risk
1    185
0    185
Name: count, dtype: int64
at_risk
1    0.5
0    0.5
Name: proportion, dtype: float64

Shape before: (280, 35) -> after: (370, 35)


## 3. Retrain XGBoost on SMOTE-balanced data

Reuses the best hyperparameters found in notebook 02 (falls back to sensible defaults if `models/best_params.json` is not found, e.g., if notebook 02 hasn't been run yet).

In [4]:
try:
    with open("models/best_params.json") as f:
        best_params = json.load(f)
    print("Loaded tuned params from notebook 02:", best_params)
except FileNotFoundError:
    best_params = {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.1}
    print("models/best_params.json not found - using default params:", best_params)

smote_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    **best_params,
)

smote_model.fit(X_train_smote, y_train_smote)
print("SMOTE-balanced model trained.")

Loaded tuned params from notebook 02: {'n_estimators': 72, 'max_depth': 3, 'learning_rate': 0.011662890273931383, 'subsample': 0.7301321323053057, 'colsample_bytree': 0.7554709158757928, 'min_child_weight': 3, 'gamma': 4.143687545759647}
SMOTE-balanced model trained.


## 4. Evaluate on the (untouched, imbalanced) test set

In [5]:
def evaluate_model(model, X, y, label="model"):
    preds = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
    }
    print(f"--- {label} ---")
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")
    print("\nConfusion matrix:")
    print(confusion_matrix(y, preds))
    print("\nClassification report:")
    print(classification_report(y, preds, zero_division=0))
    return metrics

smote_metrics = evaluate_model(smote_model, X_test, y_test, "SMOTE-balanced XGBoost (test set)")

--- SMOTE-balanced XGBoost (test set) ---
  accuracy: 0.7857
 precision: 0.6800
    recall: 0.7083
        f1: 0.6939
   roc_auc: 0.8795

Confusion matrix:
[[38  8]
 [ 7 17]]

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.83      0.84        46
           1       0.68      0.71      0.69        24

    accuracy                           0.79        70
   macro avg       0.76      0.77      0.76        70
weighted avg       0.79      0.79      0.79        70



## 5. Compare: imbalanced baseline vs SMOTE-balanced

Re-loads the baseline model from notebook 02 (if available) to produce a side-by-side comparison table. Pay particular attention to **recall** for the at-risk class — improving detection of at-risk trainees (minority class) is the primary motivation for SMOTE.

In [6]:
import os

comparison_rows = []

if os.path.exists("models/xgb_baseline.json"):
    baseline_model = xgb.XGBClassifier()
    baseline_model.load_model("models/xgb_baseline.json")
    baseline_metrics = evaluate_model(baseline_model, X_test, y_test, "Reloaded baseline (no SMOTE)")
    comparison_rows.append({"model": "baseline (no SMOTE)", **baseline_metrics})
else:
    print("models/xgb_baseline.json not found - run notebook 02 first for full comparison.")

comparison_rows.append({"model": "SMOTE-balanced", **smote_metrics})
comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df

--- Reloaded baseline (no SMOTE) ---
  accuracy: 0.7714
 precision: 0.7222
    recall: 0.5417
        f1: 0.6190
   roc_auc: 0.8723

Confusion matrix:
[[41  5]
 [11 13]]

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.89      0.84        46
           1       0.72      0.54      0.62        24

    accuracy                           0.77        70
   macro avg       0.76      0.72      0.73        70
weighted avg       0.77      0.77      0.76        70



,accuracy,precision,recall,f1,roc_auc
model,,,,,
baseline (no SMOTE),0.771429,0.722222,0.541667,0.619048,0.872283
SMOTE-balanced,0.785714,0.680000,0.708333,0.693878,0.879529


## 6. Save outputs

In [7]:
smote_model.save_model("models/xgb_smote.json")

train_smote_df = pd.DataFrame(X_train_smote, columns=X_train.columns)
train_smote_df[TARGET_COL] = y_train_smote
train_smote_df.to_csv("data/train_smote.csv", index=False)

print("Saved models/xgb_smote.json and data/train_smote.csv")

Saved models/xgb_smote.json and data/train_smote.csv


## Smoke test checklist
- [ ] SMOTE resampling runs without errors and balances class counts
- [ ] SMOTE-balanced model trains and evaluates without errors
- [ ] Comparison table displays baseline vs SMOTE metrics
- [ ] `models/xgb_smote.json` and `data/train_smote.csv` are created

Proceed to **04_shap_explainability.ipynb**.